# minillm 클라우드 학습 (Kaggle / Colab 무료 GPU)

로컬(CPU)에서는 코드 검증만 하고, **실제 학습은 이 노트북에서 무료 GPU로** 돌린다.

**Kaggle 사용법** (권장 — 주 30시간 무료, 세션 12h):
1. kaggle.com → Code → New Notebook
2. 우측 Settings → Accelerator → **GPU T4 x2** (또는 P100)
3. 이 노트북 셀을 위에서부터 실행
4. 세션이 끊기면 마지막 `--resume` 셀만 다시 실행하면 이어서 학습된다.

> `GITHUB_REPO`를 본인 저장소 주소로 바꿔라. 체크포인트는 세션 종료 시
> 사라지므로, 중간중간 `/kaggle/working`의 `checkpoints/`를 **Output으로 저장**하거나
> 마지막 셀로 다운로드해 로컬에 보관한다.

In [ ]:
GITHUB_REPO = 'https://github.com/<YOUR_ID>/minillm.git'  # <-- 본인 저장소로 변경
!git clone $GITHUB_REPO
%cd minillm
!pip install -q regex datasets tqdm
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. 데이터 준비 (최초 1회)
위키 다운로드 → 토크나이저 학습 → 토큰화(.bin). 처음엔 `--max-docs`로 작게 시험하고,
잘 되면 인자를 빼고 전체로 돌린다.

In [ ]:
# 전체를 받으려면 --max-docs 를 지운다 (위키 ko 전체 ≈ 1.4GB, 15~30분)
!python -m data.download --max-docs 50000
!python -m tokenizer.train_tokenizer --input data/raw/corpus.txt --vocab-size 16384 --sample-mb 200
!python -m data.pack --input data/raw/corpus.txt --tokenizer tokenizer/tokenizer.json

## 2. 사전학습
T4 기준 대략 1500~2500 step/시간. `max_steps`는 `train/config.py`의 full 프리셋(기본 40000).
세션이 끊기면 아래 셀을 `--resume` 붙여 다시 실행.

In [ ]:
!python -m train.pretrain --preset full            # 처음 시작
# !python -m train.pretrain --preset full --resume  # 세션 재개 시 이 줄

## 3. 대화 파인튜닝 (SFT)
사전학습 best 체크포인트를 이어받아 대화 형식을 가르친다. 30분~1시간.

In [ ]:
!python -m data.prepare_sft --tokenizer tokenizer/tokenizer.json
!python -m train.sft --init checkpoints/ckpt_best.pt --data data/bin/sft.npz --out checkpoints/sft.pt

## 4. 체크포인트 저장
`checkpoints/sft.pt`와 `tokenizer/tokenizer.json`을 로컬로 내려받아
`python chat.py --ckpt checkpoints/sft.pt`로 대화한다. (Kaggle은 오른쪽 Output 패널에서 다운로드)

In [ ]:
import shutil, os
os.makedirs('/kaggle/working/out', exist_ok=True)
for f in ['checkpoints/sft.pt', 'checkpoints/ckpt_best.pt', 'tokenizer/tokenizer.json']:
    if os.path.exists(f):
        shutil.copy(f, '/kaggle/working/out/')
print('저장 완료 — Output 패널에서 다운로드하세요')